In [3]:
import logging
import threading
from abc import ABC, abstractmethod
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Iterator, Optional, Type
import json
from tqdm.notebook import tqdm
import httpx
import polars as pl
import pyarrow.parquet as pq
import yaml
import zstandard as zstd
import time
import queue
import multiprocessing as mp
from dataclasses import dataclass
from typing import Any, Iterable, List, Optional, Tuple

import experiment.main_registry
from data_connectors.mmlu_pro import MMLUProExample, MMLUProCategory
from experiment.main_registry import DataConnectorName
from concurrent.futures import ThreadPoolExecutor, Future
from pydantic import BaseModel, ConfigDict
from utils.hydra_config import MainConfig
from utils.tracking import TrackEntry
import pyarrow

%load_ext autoreload
%autoreload 2

In [4]:
PROJECT_PATH = Path(
    "/Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/social_studies/results/runs/standard/2026-01-08-09-49-55")
JSONL_PATH = PROJECT_PATH / "experiment_result.jsonl.zst"

PHOENIX_ENDPOINT = "http://localhost:6006"


In [5]:
def read_hydra_config(run_dir: Path) -> MainConfig:
    """
    Common Hydra output: <run_dir>/.hydra/config.yaml (and overrides.yaml).
    Adjust paths for your setup.
    """
    cfg_path = run_dir / ".hydra" / "config.yaml"
    if not cfg_path.exists():
        raise FileNotFoundError(cfg_path)

    config = yaml.safe_load(cfg_path.read_text(encoding="utf-8"))

    return MainConfig.model_validate(config, strict=False)


def read_log_text(run_dir: Path, log_filename: str = "main.log") -> Optional[str]:
    p = run_dir / log_filename
    if not p.exists():
        return None
    return p.read_text(encoding="utf-8", errors="replace")


def iter_jsonl_zst(path: Path) -> Iterator[Dict[str, Any]]:
    """
    Streams a .jsonl.zst file and yields dict per line.
    """
    with path.open("rb") as f:
        dctx = zstd.ZstdDecompressor()
        with dctx.stream_reader(f) as reader:
            buf = b""
            while True:
                chunk = reader.read(1 << 20)
                if not chunk:
                    break
                buf += chunk
                while b"\n" in buf:
                    line, buf = buf.split(b"\n", 1)
                    if not line.strip():
                        continue
                    yield json.loads(line)
            if buf.strip():
                yield json.loads(buf)

In [6]:
def get_span_attributes(phoenix_graphql_endpoint: str, *, spanIds: list[str]):
    fields = "\n".join(
        f's_{oid}: getSpanByOtelId(spanId: "{oid}") {{ attributes }}'
        for oid in spanIds
    )
    query = f"query GetSpans {{\n{fields}\n}}"

    with httpx.Client() as client:
        response = client.post(
            phoenix_graphql_endpoint,
            json={
                "query": query,
            },
            headers={
                "Content-Type": "application/json"
            }
        )

    response.raise_for_status()
    data = response.json()

    if "errors" in data:
        raise RuntimeError(data["errors"])

    out = {}
    for alias, node in data["data"].items():
        original_oid = alias[2:]
        if node is None:
            out[original_oid] = None
            continue

        attrs = node["attributes"]
        span = json.loads(attrs) if isinstance(attrs, str) else attrs
        out[original_oid] = span

    return out


In [7]:
import ast


class PolarsBaseModel(BaseModel, ABC):

    @classmethod
    @abstractmethod
    def from_raw_data(cls, assigned_id: int, entry: TrackEntry, hydra_config: MainConfig,
                      span_attributes: dict[str, Any]) -> "PolarsBaseModel":
        ...

    @staticmethod
    @abstractmethod
    def get_polars_schema() -> pl.Schema:
        ...


class Question(PolarsBaseModel):
    model_config = ConfigDict(use_enum_values=True)
    question_id: int

    original_question_id: int

    data_connector: DataConnectorName
    original_source: str
    category: str
    question: str
    answer_options: list[str]
    answer_index: int
    answer_string: str

    @classmethod
    def from_raw_data(cls, assigned_id: int, entry: TrackEntry, hydra_config: MainConfig,
                      span_attributes: dict[str, Any]) -> "Question":
        match hydra_config.experiment.data.value:
            case "mmlu-pro":
                # TODO: Remove this when new data is generated
                if not isinstance(options := span_attributes["question"]["options"], list):
                    span_attributes["question"]["options"] = ast.literal_eval(options)

                if not isinstance(category := span_attributes["question"]["category"], MMLUProCategory):
                    span_attributes["question"]["category"] = MMLUProCategory[category.split(".")[-1]]

                question = MMLUProExample.model_validate(span_attributes["question"])
                return cls(
                    question_id=assigned_id,
                    original_question_id=question.question_id,
                    data_connector=hydra_config.experiment.data,
                    original_source=question.src,
                    category=question.category.value,
                    question=entry.input.question,
                    answer_options=question.options,
                    answer_index=question.answer_index,
                    answer_string=question.answer,
                )
            case _:
                raise NotImplementedError("Data Connector not Implemented yet")

    @staticmethod
    def get_polars_schema() -> pl.Schema:
        return pl.Schema({
            "question_id": pl.UInt32,
            "original_question_id": pl.UInt32,
            "data_connector": pl.Utf8,
            "original_source": pl.String,
            "category": pl.String,
            "question": pl.String,
            "answer_options": pl.List(pl.String),
            "answer_index": pl.UInt8,
            "answer_string": pl.String,
        })


In [8]:
def polars_schema_to_arrow_schema(polars_schema: pl.Schema) -> pyarrow.Schema:
    return pl.DataFrame(schema=polars_schema).to_arrow().schema

In [9]:
MAX_PARALLEL_REQUESTS = 4
MAX_IDS_PER_REQUEST = 200
IN_QUEUE_MAXSIZE = 1000
OUT_QUEUE_MAXSIZE = 1000
CHUNK_SIZE = 1000

QUEUE_TIMEOUT = 10  #seconds

SENTINEL = "___SENTINEL___"

logger = logging.getLogger("PARQUET BUILDER")


@dataclass(frozen=True)
class Package:
    id: int
    span_id: str
    entry: TrackEntry
    span_info: dict[str, Any] | None = None


def main_process_loop(
        in_q: queue.Queue,
        out_q: queue.Queue,
        pool: ThreadPoolExecutor,
):
    futures: set[Future[dict[str, Any]]] = set()
    shutting_down = False

    in_flight = 0
    in_flight_lock = threading.Lock()
    all_done = threading.Event()

    def submit_batch_call(batch: list[Package]):
        nonlocal in_flight
        with in_flight_lock:
            in_flight += 1

        f = pool.submit(get_span_attributes, phoenix_graphql_endpoint=PHOENIX_ENDPOINT + "/graphql",
                        spanIds=[b.span_id for b in batch])
        futures.add(f)

        def _done_callback(fut: Future) -> None:
            nonlocal in_flight
            nonlocal shutting_down
            try:
                span_infos = fut.result()
                logger.warning(f'Received batch with {len(span_infos)} spans')
                for b in batch:
                    out_q.put(Package(id=b.id, span_id=b.span_id, entry=b.entry, span_info=span_infos[b.span_id]))

            except httpx.ConnectError:
                logger.error("Connection error, stopping procedure.")
                shutting_down = True
                all_done.set()
            except Exception as e:
                raise NotImplementedError()
            finally:
                futures.discard(fut)
                with in_flight_lock:
                    in_flight -= 1
                    if shutting_down and not in_flight:
                        all_done.set()

        f.add_done_callback(_done_callback)

    pending: List[Package] = []

    while True:
        while (not shutting_down) and len(pending) < MAX_IDS_PER_REQUEST:

            try:
                item = in_q.get(timeout=QUEUE_TIMEOUT)
            except queue.Empty:
                break
            except TimeoutError:
                break

            if item == SENTINEL:
                shutting_down = True
                break
            pending.append(item)

        if pending and (len(pending) >= MAX_IDS_PER_REQUEST or shutting_down):
            batch = pending[:MAX_IDS_PER_REQUEST]
            pending = pending[MAX_IDS_PER_REQUEST:]
            submit_batch_call(batch)
            continue

        if shutting_down:
            while futures:
                fut = futures.pop()
                try:
                    fut.result(timeout=QUEUE_TIMEOUT)
                except TimeoutError:
                    futures.add(fut)

            all_done.wait()
            out_q.put(SENTINEL)
            return


def drain_results_nonblocking(out_q: queue.Queue, buffer: List[Package]) -> None:
    for _ in range(MAX_IDS_PER_REQUEST):
        try:
            msg = out_q.get_nowait()
        except queue.Empty:
            return

        if msg == SENTINEL:
            out_q.put(SENTINEL)
            return

        buffer.append(msg)


async def build_questions_parquet(
        consumable: Iterator[Dict[str, Any]],
        output_parquet: str,
        config: MainConfig,
):
    pl_schema = Question.get_polars_schema()
    arrow_schema = polars_schema_to_arrow_schema(pl_schema)

    writer: pq.ParquetWriter = pq.ParquetWriter(
        output_parquet,
        arrow_schema,
        compression="zstd",
        use_dictionary=True,
        write_statistics=True,
    )
    pool = ThreadPoolExecutor(max_workers=MAX_PARALLEL_REQUESTS)

    buffer: list[Package] = []

    def flush(buf: list[Package]) -> None:
        logger.warning(f"flush buffer with {len(buf)} items")

        nonlocal writer

        items = [Question.from_raw_data(b.id, b.entry, config, b.span_info).model_dump() for b in buf]

        df = pl.DataFrame(items, schema=pl_schema, orient="row")
        table = df.to_arrow()
        table = table.cast(arrow_schema)
        writer.write_table(table)

    in_q: queue.Queue = queue.Queue(maxsize=IN_QUEUE_MAXSIZE)
    out_q: queue.Queue = queue.Queue(maxsize=OUT_QUEUE_MAXSIZE)

    thread = threading.Thread(
        target=main_process_loop,
        args=(in_q, out_q, pool),
        daemon=True,
    )

    thread.start()

    for _id, item in tqdm(enumerate(consumable)):
        tracked = TrackEntry.model_validate(item)
        in_q.put(Package(id=_id, span_id=tracked.phoenix_span_info.span_id_hex, entry=tracked, span_info=None))

        drain_results_nonblocking(out_q, buffer)

    in_q.put(SENTINEL)

    while (msg := out_q.get()) != SENTINEL:
        buffer.append(msg)
        if len(buffer) >= CHUNK_SIZE:
            flush(buffer)
            buffer.clear()

    if buffer:
        flush(buffer)

    thread.join(timeout=5)
    pool.shutdown(wait=False)
    writer.close()

In [10]:
hydra_config = read_hydra_config(PROJECT_PATH)


In [11]:
await build_questions_parquet(iter_jsonl_zst(Path(JSONL_PATH)), "questions.parquet", hydra_config)

0it [00:00, ?it/s]

Received batch with 200 spans
Received batch with 200 spans
Received batch with 200 spans
Received batch with 200 spans
Received batch with 200 spans
flush buffer with 1000 items
Received batch with 200 spans
Received batch with 200 spans
Received batch with 200 spans
Received batch with 200 spans
Received batch with 200 spans
flush buffer with 1000 items
Received batch with 200 spans
Received batch with 200 spans
Received batch with 200 spans
Received batch with 200 spans
Received batch with 200 spans
flush buffer with 1000 items
Received batch with 200 spans
Received batch with 200 spans
Received batch with 200 spans
Received batch with 200 spans
Received batch with 200 spans
flush buffer with 1000 items
Received batch with 200 spans
Received batch with 200 spans
Received batch with 200 spans
Received batch with 200 spans
Received batch with 200 spans
flush buffer with 1000 items
Received batch with 200 spans
Received batch with 200 spans
Received batch with 200 spans
Received batch 

In [13]:
data = pl.read_parquet("questions.parquet")

In [14]:
data

question_id,original_question_id,data_connector,original_source,category,question,answer_options,answer_index,answer_string
u32,u32,str,str,str,str,list[str],u8,str
200,264,"""mmlu-pro""","""ori_mmlu-marketing""","""business""","""Q: On August 4, a store purcha…","[""$6,300 "", ""$7,200"", … ""$6,500""]",6,"""G"""
201,279,"""mmlu-pro""","""stemez-Business""","""business""","""Q: Montgomery's Department Sto…","[""$100"", ""$67.50"", … ""$75""]",6,"""G"""
202,280,"""mmlu-pro""","""stemez-Business""","""business""","""Q: Mr. Charles owns a brick bu…","[""$95 per year"", ""$110 per year"", … ""$120 per year""]",5,"""F"""
203,275,"""mmlu-pro""","""stemez-Business""","""business""","""Q: Bill deposits $1,000 for 4 …","[""$1,000"", ""$1,102.50"", … ""$1,300""]",2,"""C"""
204,281,"""mmlu-pro""","""stemez-Business""","""business""","""Q: The distributors of oil fro…","[""Equilibrium price is 18 dollars, Number of barrels sold is 54 per day"", ""Equilibrium price is 12 dollars, Number of barrels sold is 72 per day"", … ""Equilibrium price is 20 dollars, Number of barrels sold is 40 per day""]",9,"""J"""
…,…,…,…,…,…,…,…,…
12027,12248,"""mmlu-pro""","""stemez-TransportPhenomena""","""engineering""","""Q: Water at 340°K and a rate o…","[""Length of piping required is 2m and the maximum temperature at the exit is 360K"", ""Length of piping required is 1.48m and the maximum temperature at the exit is 354.8K"", … ""Length of piping required is 1.75m and the maximum temperature at the exit is 360K""]",1,"""B"""
12028,12249,"""mmlu-pro""","""stemez-TransportPhenomena""","""engineering""","""Q: A solid sphere of naphthale…","[""2.22 × 10^-8kgmol/ m^2-sec"", ""1.67 × 10^-8 kgmol/ m^2-sec"", … ""3.05 × 10^-8 kgmol/ m^2-sec""]",3,"""D"""
12029,12245,"""mmlu-pro""","""stemez-TransportPhenomena""","""engineering""","""Q: Oil of viscosity 0.2248lbm/…","[""Average velocity of flow: 2.26 ft/s, Power consumed: 8.61 hp/mile, Velocity at 3/2 in. from center: 1.98 ft/s, Shear stress at 3/2 in. from center: 0.284 lb/ft^2"", ""Average velocity of flow: 3.0 ft/s, Power consumed: 10.2 hp/mile, Velocity at 3/2 in. from center: 2.5 ft/s, Shear stress at 3/2 in. from center: 0.35 lb/ft^2"", … ""Average velocity of flow: 1.8 ft/s, Power consumed: 7.2 hp/mile, Velocity at 3/2 in. from center: 1.4 ft/s, Shear stress at 3/2 in. from center: 0.22 lb/ft^2""]",0,"""A"""


In [26]:
from phoenix.client import AsyncClient, Client

client = Client(base_url=PHOENIX_ENDPOINT)
async_client = AsyncClient(base_url=PHOENIX_ENDPOINT)

In [29]:
# TODO: Change when code changed:
# project_name = used_hydra_config.meta_info.phoenix_project_name
project_name = PROJECT_PATH.parent.name + " - " + PROJECT_PATH.name
print(project_name)

spans = client.spans.get_spans(project_identifier=project_name)

standard - 2026-01-08-09-49-55


In [37]:
ids = [span["context"]["span_id"] for span in spans]
ids

['5b71d325eb1409e8',
 '1b73d8f3095d465e',
 '3c2b6e1e6f84aef8',
 '1c9be92e0110b1d0',
 'f2acca75220380d5',
 '2db55f744b815743',
 '573c96c98b30b35a',
 '7bb25daa443c823b',
 'a133c3a4231bb07d',
 '18c89d52a858ad4c',
 '75b17b838b8aa1fc',
 '361f3729232a1ff6',
 '922c152e41f32a8e',
 '21fde4be3fefeace',
 'fd12b8ef52eaad14',
 'b8eb4bda818c6c2c',
 'e80487cadf3dcdcc',
 '910d3c67dae442e7',
 '12dd07e3bfb0fc35',
 'dcf618236a40986c',
 '705476ddf5da2c4b',
 'd993b5d5745949b4',
 '8fa0db9d69a9384a',
 '9e63f64422d55b5f',
 '31dac57726d10c29',
 '1975af7895c437c6',
 '36ab78d91a83938f',
 'b2a22bcd93020c6e',
 '9412ee011aef85b7',
 '725e2cb3c31b0f1d',
 'e5f539491a353d5f',
 'd11b4f3353fa2925',
 '9b4698574eb0acbc',
 '997bccc45689b4c0',
 '5412b5d298dff436',
 '37aa3125c9efa137',
 'eb9b071100628614',
 '530c32419bb53c38',
 'c4be253611c5f7e7',
 'ed711c5a98ce80d6',
 '9b4ae168067cf5a9',
 'dd8eb0a9333d267e',
 '6889e6c73d167016',
 'e2de08d904674236',
 'aa11cf53da2f1ac9',
 '81f528b0380a4bd2',
 'b9aa5fab99296248',
 '5488926f4a2

In [36]:
await get_span_attributes(PHOENIX_ENDPOINT + "/graphql", spanIds=ids)

{'5b71d325eb1409e8': {'openinference': {'span': {'kind': 'chain'}},
  'experiment': {'example_id': 12255},
  'output': {'answers_at_end': 'None',
   'answers_at_beginning': 'None',
   'final_answer': "To compute the local friction coefficient, we use the formula:\n\n\\[ \\alpha = 9 \\frac{F}{I d^2} - 9 \\frac{\\left(\\frac{dF}{R}\\right)}{I d} + \\frac{F^2}{4\\tau r^3} \\]\n\nWhere:\n- \\( \\alpha \\) is the local friction coefficient\n- \\( F \\) is the force per unit length\n- \\( I \\) is the current (in Amps if you calculate it using Ampere-turns)\n- \\( d \\) is the hydraulic diameter (in meters if you calculate it using meters)\n- \\( r \\) is the radius of the tube (in meters if you calculate it using meters)\n- \\( \\tau \\) is the static frictional retardation coefficient\n- \\( \\alpha \\) is positive for anti-uniform flow\n\nGiven:\n- \\( d = 0.2 \\) m\n- \\( r = 0.07 \\) m (radius in meters)\n- \\( \\delta F \\) is the dimensionless Shuttleworth coefficient, 1.4\n\nLet's ca

In [12]:
await async_client.projects.list()

[{'name': 'multi-agent-debate - 2026-01-09-07-37-38',
  'description': None,
  'id': 'UHJvamVjdDoxMg=='},
 {'name': 'standard - 2026-01-08-09-49-55',
  'description': None,
  'id': 'UHJvamVjdDoxMQ=='},
 {'name': 'default', 'description': 'Default project', 'id': 'UHJvamVjdDox'}]

In [13]:
x = TrackEntry.model_validate(next(iter_jsonl_zst(Path(JSONL_PATH))))

In [15]:
client.spans.get_span_annotations(span_ids=[x.phoenix_span_info.span_id_hex], project_identifier=project_name)

[]

In [27]:
spans = client.spans.get_spans(project_identifier=project_name, timeout=60, limit=100)

NameError: name 'project_name' is not defined

In [22]:
from utils.general import unflatten_dict

unflatten_dict(spans[0])["attributes"]["output"].keys()

dict_keys(['answers_at_end', 'answers_at_beginning', 'final_answer', 'used_output_tokens', 'used_input_tokens'])

In [58]:
for x in iter_jsonl_zst(Path(JSONL_PATH)):
    x = TrackEntry.model_validate(x)
    break

In [64]:
from pprint import pprint
import json

a = await get_span_attributes(x.phoenix_span_info.span_id_hex, PHOENIX_ENDPOINT + "/graphql")

In [65]:
a

{'openinference': {'span': {'kind': 'chain'}},
 'experiment': {'example_id': 70},
 'output': {'answers_at_end': 'None',
  'answers_at_beginning': 'None',
  'final_answer': 'The answer is (H).',
  'used_output_tokens': 7,
  'used_input_tokens': 1044},
 'question': {'question_id': 70,
  'src': 'ori_mmlu-business_ethics',
  'category': 'MMLUProCategory.BUSINESS',
  'answer_index': 8,
  'options': "['Safe practices, Fear, Jealousy, Trivial', 'Unsafe practices, Distress, Joy, Trivial', 'Safe practices, Wants, Jealousy, Trivial', 'Safe practices, Distress, Fear, Trivial', 'Unsafe practices, Wants, Jealousy, Serious', 'Safe practices, Distress, Jealousy, Serious', 'Safe practices, Wants, Fear, Serious', 'Unsafe practices, Wants, Fear, Trivial', 'Unsafe practices, Distress, Fear, Serious']",
  'question': 'Typical advertising regulatory bodies suggest, for example that adverts must not: encourage _________, cause unnecessary ________ or _____, and must not cause _______ offence.',
  'answer': 